<a href="https://colab.research.google.com/github/Alessandro-json/AI_PostProcessing_Detection/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2 - Joint Detection of AI-Generated Images and Post-Processing Alterations in Real-World Scenarios

# Imports

In [2]:
import os
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image


# Globals


In [3]:
# Choose where the project will be stored in Colab.
WORKDIR = Path('/content')

REPO_URL = 'https://github.com/Alessandro-json/AI_PostProcessing_Detection'

# Repository folder name after git clone.
REPO_DIR = WORKDIR / 'REPO'

# Main paths used by the scripts.
TRAIN_CSV = 'data/splits/train_balanced.csv'
VAL_CSV = 'data/splits/val_balanced.csv'
TEST_CSV = 'data/splits/test_balanced.csv'

IMAGE_ROOT = "/content/data/raw/RRDataset_subset"
CHECKPOINT_NAME = 'best_rgb.pt'
CHECKPOINT_PATH = f'checkpoints/{CHECKPOINT_NAME}'
DEPTH_ROOT = "/content/drive/MyDrive/CV_Project/depth_maps"

DATASET_FILE_ID = "1HJamsAB4Lj90xNjA6tkfIrstHGJqklsI"
DATASET_ZIP_PATH = "/content/RRDataset_subset.zip"

# Training hyperparameters for the first baseline.
EPOCHS = 10
BATCH_SIZE = 32
IMAGE_SIZE = 224
NUM_WORKERS = 2
LEARNING_RATE = 1e-4

# Multi-task loss weights.
LAMBDA_FAKE = 1.0
LAMBDA_TRANSFORM = 1.0


# Setup repository



In [4]:
%cd {WORKDIR}

if REPO_DIR.exists():
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}


/content
Cloning into '/content/REPO'...
remote: Enumerating objects: 198, done.
remote: Counting objects: 100% (198/198), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 198 (delta 107), reused 73 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (198/198), 12.72 MiB | 11.59 MiB/s, done.
Resolving deltas: 100% (107/107), done.
/content/REPO


# Install dependencies

In [5]:
# Install project dependencies.
!pip install -q -r requirements.txt


# Utils

In [6]:
def show_csv_summary(csv_path):
    """Print a quick summary of one split CSV."""
    path = Path(csv_path)
    if not path.exists():
        print(f'Missing file: {path}')
        return None

    df = pd.read_csv(path)
    print(f'File: {path}')
    print(f'Rows: {len(df)}')
    print('Columns:', list(df.columns))

    if 'fake_label' in df.columns:
        print('\nFake label distribution:')
        print(df['fake_label'].value_counts().sort_index())

    if 'transform_label' in df.columns:
        print('\nTransform label distribution:')
        print(df['transform_label'].value_counts().sort_index())

    if {'fake_label', 'transform_label'}.issubset(df.columns):
        print('\nJoint distribution:')
        print(pd.crosstab(df['transform_label'], df['fake_label'], rownames=['transform'], colnames=['fake']))

    return df


In [7]:
def show_image_exists_check(df, image_root, n=5):
    """Check whether the first n image paths exist."""
    if df is None:
        return

    root = Path(image_root)
    print(f'Image root: {root}')

    for rel_path in df['image_path'].head(n):
        full_path = root / rel_path
        print(full_path, 'OK' if full_path.exists() else 'MISSING')

In [8]:
def show_evaluation_outputs(output_dir):
    """
    Show evaluation metrics and confusion matrices saved by evaluate_RGB.py.
    """

    output_dir = Path(output_dir)
    metrics_path = output_dir / "metrics.json"

    if not metrics_path.exists():
        print(f"Metrics file not found: {metrics_path}")
        return

    # Load metrics saved by evaluate_RGB.py.
    with open(metrics_path, "r", encoding="utf-8") as f:
        metrics = json.load(f)

    # Convert metrics dictionary into a readable table.
    rows = []

    for metric_name, value in metrics.items():
        if isinstance(value, dict):
            for sub_metric_name, sub_value in value.items():
                rows.append({
                    "Metric": metric_name,
                    "Group": sub_metric_name,
                    "Value": sub_value,
                })
        else:
            rows.append({
                "Metric": metric_name,
                "Group": "-",
                "Value": value,
            })

    metrics_df = pd.DataFrame(rows)

    # Format numeric values to 4 decimals.
    if not metrics_df.empty:
        metrics_df["Value"] = metrics_df["Value"].apply(
            lambda x: f"{x:.4f}" if isinstance(x, (int, float)) else x
        )

    print("Evaluation metrics")
    display(metrics_df)

    # Show saved confusion matrices as images.
    fake_cm_path = output_dir / "confusion_fake.png"
    transform_cm_path = output_dir / "confusion_transform.png"

    if fake_cm_path.exists():
        print("\nReal/Fake confusion matrix")
        display(Image(filename=str(fake_cm_path)))

    if transform_cm_path.exists():
        print("\nTransformation confusion matrix")
        display(Image(filename=str(transform_cm_path)))

    if not fake_cm_path.exists() and not transform_cm_path.exists():
        print("\nNo confusion matrix images found in this folder.")

# Data


In [9]:
train_df = show_csv_summary(TRAIN_CSV)

File: data/splits/train_balanced.csv
Rows: 2100
Columns: ['image_path', 'fake_label', 'transform_label', 'category', 'transform_name', 'fake_name']

Fake label distribution:
fake_label
0    1050
1    1050
Name: count, dtype: int64

Transform label distribution:
transform_label
0    700
1    700
2    700
Name: count, dtype: int64

Joint distribution:
fake         0    1
transform          
0          350  350
1          350  350
2          350  350


In [10]:
val_df = show_csv_summary(VAL_CSV)

File: data/splits/val_balanced.csv
Rows: 450
Columns: ['image_path', 'fake_label', 'transform_label', 'category', 'transform_name', 'fake_name']

Fake label distribution:
fake_label
0    225
1    225
Name: count, dtype: int64

Transform label distribution:
transform_label
0    150
1    150
2    150
Name: count, dtype: int64

Joint distribution:
fake        0   1
transform        
0          75  75
1          75  75
2          75  75


In [11]:
test_df = show_csv_summary(TEST_CSV)

File: data/splits/test_balanced.csv
Rows: 450
Columns: ['image_path', 'fake_label', 'transform_label', 'category', 'transform_name', 'fake_name']

Fake label distribution:
fake_label
0    225
1    225
Name: count, dtype: int64

Transform label distribution:
transform_label
0    150
1    150
2    150
Name: count, dtype: int64

Joint distribution:
fake        0   1
transform        
0          75  75
1          75  75
2          75  75


In [12]:
# Download the dataset only if the zip file is not already available.
if not Path(DATASET_ZIP_PATH).exists():
    print("Dataset zip not found. Downloading it with gdown...")
    os.system(f'gdown --id "{DATASET_FILE_ID}" -O "{DATASET_ZIP_PATH}"')
else:
    print(f"Dataset zip already exists: {DATASET_ZIP_PATH}")

Dataset zip not found. Downloading it with gdown...


In [13]:
# Extract the dataset only if the extracted folder is not already available.
IMAGE_ROOT = Path(IMAGE_ROOT)
if not IMAGE_ROOT.exists():
    print("Extracted dataset folder not found. Extracting...")
    os.system(f'mkdir -p "{IMAGE_ROOT.parent}"')
    os.system(f'unzip -q "{DATASET_ZIP_PATH}" -d "{IMAGE_ROOT.parent}"')
else:
    print(f"Dataset already extracted: {IMAGE_ROOT}")

Extracted dataset folder not found. Extracting...


In [14]:
show_image_exists_check(train_df, IMAGE_ROOT, n=5)

Image root: /content/data/raw/RRDataset_subset
/content/data/raw/RRDataset_subset/original/real/real_006970.jpg OK
/content/data/raw/RRDataset_subset/original/real/real_003543.jpg OK
/content/data/raw/RRDataset_subset/original/real/real_004687.jpg OK
/content/data/raw/RRDataset_subset/original/real/real_001434.jpg OK
/content/data/raw/RRDataset_subset/original/real/real_004760.jpg OK


# RGB baseline
## Train and Evaluation
### Single-task Real/Fake Baseline

In [15]:
# Train the real/fake single-task baseline.
!python src/train_RGB.py \
  --task fake \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --checkpoint_name best_rgb_fake.pt

Traceback (most recent call last):
  File "/content/REPO/src/train_RGB.py", line 9, in <module>
    from dataset import RRDatasetFromCSV, build_train_transform, build_eval_transform
  File "/content/REPO/src/dataset.py", line 8, in <module>
    from torchvision import transforms
  File "/usr/local/lib/python3.12/dist-packages/torchvision/__init__.py", line 8, in <module>
    from torchvision import _meta_registrations, datasets, io, models, ops, transforms, utils  # usort:skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchvision/models/__init__.py", line 2, in <module>
    from .convnext import *
  File "/usr/local/lib/python3.12/dist-packages/torchvision/models/convnext.py", line 9, in <module>
    from ..ops.misc import Conv2dNormActivation, Permute
  File "/usr/local/lib/python3.12/dist-packages/torchvision/ops/__init__.py", line 23, in <module>
    from .poolers import MultiScaleRoI

In [16]:
# Evaluate the real/fake single-task baseline.
!python src/evaluate_RGB.py \
  --task fake \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint checkpoints/best_rgb_fake.pt \
  --output_dir results/rgb_fake \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

Traceback (most recent call last):
  File "/content/REPO/src/evaluate_RGB.py", line 8, in <module>
    from sklearn.metrics import (
  File "/usr/local/lib/python3.12/dist-packages/sklearn/__init__.py", line 73, in <module>
    from .base import clone  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 19, in <module>
    from .utils._estimator_html_repr import _HTMLDocumentationLinkMixin, estimator_html_repr
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/__init__.py", line 15, in <module>
    from ._chunking import gen_batches, gen_even_slices
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/_chunking.py", line 11, in <module>
    from ._param_validation import Interval, validate_params
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/_param_validation.py", line 17, in <module>
    from .validation import _is_arraylike_not_scalar
  File "/usr/local/lib/python3.12/dist-packages/sklearn

In [17]:
show_evaluation_outputs("results/rgb_fake")

Metrics file not found: results/rgb_fake/metrics.json


### Single-task Transformation Baseline

In [18]:
# Train the transformation single-task baseline.
!python src/train_RGB.py \
  --task transform \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --checkpoint_name best_rgb_transform.pt

object address  : 0x7d7cc63aa500
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
Traceback (most recent call last):
^C


In [19]:
# Evaluate the transformation single-task baseline.
!python src/evaluate_RGB.py \
  --task transform \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint checkpoints/best_rgb_transform.pt \
  --output_dir results/rgb_transform \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

Traceback (most recent call last):
  File "/content/REPO/src/evaluate_RGB.py", line 7, in <module>
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2345, in <module>
    from torch import quantization as quantization  # usort: skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/quantization/__init__.py", line 2, in <module>
    from .fake_quantize import *  # noqa: F403
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/quantization/fake_quantize.py", line 10, in <module>
    from torch.ao.quantization.fake_quantize import (
  File "/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/__init__.py", line 11, in <module>
    from .fake_quantize import *  # noqa: F403
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/fake_quantize.py", line 10, in <module>
    from torch.ao.quantization.observer import (
  File 

In [20]:
show_evaluation_outputs("results/rgb_transform")

Metrics file not found: results/rgb_transform/metrics.json


### Multi-task Baseline, Manual Weights 1.0 / 1.0

In [21]:
# Train the joint RGB multi-task baseline with weights 1 1.
!python src/train_RGB.py \
  --task multitask \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lambda_fake 1.0 \
  --lambda_transform 1.0 \
  --checkpoint_name best_rgb_multitask_1_1.pt

object address  : 0x7f155f870580
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
Traceback (most recent call last):
^C


In [22]:
# Evaluate the joint RGB multi-task baseline with weights 1 1.
!python src/evaluate_RGB.py \
  --task multitask \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint checkpoints/best_rgb_multitask_1_1.pt \
  --output_dir results/rgb_multitask_1_1 \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

Traceback (most recent call last):
  File "/content/REPO/src/evaluate_RGB.py", line 7, in <module>
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2787, in <module>
    from torch import (
  File "/usr/local/lib/python3.12/dist-packages/torch/export/__init__.py", line 39, in <module>
    import torch.export.custom_ops
  File "/usr/local/lib/python3.12/dist-packages/torch/export/custom_ops.py", line 4, in <module>
    import torch
  File "<frozen importlib._bootstrap>", line 471, in _lock_unlock_module
  File "<frozen importlib._bootstrap>", line 311, in acquire
  File "<frozen importlib._bootstrap>", line 170, in __enter__
  File "<frozen importlib._bootstrap>", line 128, in setdefault
KeyboardInterrupt
^C


In [23]:
show_evaluation_outputs("results/rgb_multitask_1_1")

Metrics file not found: results/rgb_multitask_1_1/metrics.json


### Multi-task Baseline, Manual Weights 1.0 / 2.0

In [24]:
# Train the joint RGB multi-task baseline with weights 1 2.
!python src/train_RGB.py \
  --task multitask \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lambda_fake 1.0 \
  --lambda_transform 2.0 \
  --checkpoint_name best_rgb_multitask_1_2.pt

Traceback (most recent call last):
  File "/content/REPO/src/train_RGB.py", line 4, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2804, in <module>
    from torch import _meta_registrations
  File "/usr/local/lib/python3.12/dist-packages/torch/_meta_registrations.py", line 12, in <module>
    from torch._decomp import (
  File "/usr/local/lib/python3.12/dist-packages/torch/_decomp/__init__.py", line 287, in <module>
    import torch._decomp.decompositions
  File "/usr/local/lib/python3.12/dist-packages/torch/_decomp/decompositions.py", line 17, in <module>
    import torch._prims as prims
  File "/usr/local/lib/python3.12/dist-packages/torch/_prims/__init__.py", line 1195, in <module>
    remainder = _make_elementwise_binary_prim(
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/_prims/__init__.py", line 526, in _make_elementwise_binary_prim
    return _make_prim(
           ^^^^

In [25]:
# Evaluate the joint RGB multi-task baseline with weights 1 2.
!python src/evaluate_RGB.py \
  --task multitask \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint checkpoints/best_rgb_multitask_1_2.pt \
  --output_dir results/rgb_multitask_1_2 \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

Traceback (most recent call last):
  File "/content/REPO/src/evaluate_RGB.py", line 7, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2787, in <module>
    from torch import (
  File "/usr/local/lib/python3.12/dist-packages/torch/export/__init__.py", line 42, in <module>
    from .decomp_utils import CustomDecompTable
  File "/usr/local/lib/python3.12/dist-packages/torch/export/decomp_utils.py", line 5, in <module>
    from torch._export.utils import (
  File "/usr/local/lib/python3.12/dist-packages/torch/_export/__init__.py", line 49, in <module>
    from .wrappers import _wrap_submodules
  File "/usr/local/lib/python3.12/dist-packages/torch/_export/wrappers.py", line 10, in <module>
    from torch._higher_order_ops.flat_apply import (
  File "/usr/local/lib/python3.12/dist-packages/torch/_higher_order_ops/__init__.py", line 14, in <module>
    from torch._higher_order_ops.effects import with_effects
  File "/usr/local/lib/python3

In [26]:
show_evaluation_outputs("results/rgb_multitask_1_2")

Metrics file not found: results/rgb_multitask_1_2/metrics.json


### Multi-task Baseline, Manual Weights 2.0 / 1.0

In [27]:
# Train the joint RGB multi-task baseline with weights 2 1.
!python src/train_RGB.py \
  --task multitask \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lambda_fake 2.0 \
  --lambda_transform 1.0 \
  --checkpoint_name best_rgb_multitask_2_1.pt

object address  : 0x7d9b0a7051e0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
Traceback (most recent call last):
^C


In [28]:
# Evaluate the joint RGB multi-task baseline with weights 2 1.
!python src/evaluate_RGB.py \
  --task multitask \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint checkpoints/best_rgb_multitask_2_1.pt \
  --output_dir results/rgb_multitask_2_1 \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

Traceback (most recent call last):
  File "/content/REPO/src/evaluate_RGB.py", line 7, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2273, in <module>
    from torch import (
  File "/usr/local/lib/python3.12/dist-packages/torch/nested/__init__.py", line 21, in <module>
    from ._internal.nested_tensor import _rebuild_njt, NestedTensor as _NestedTensor
  File "/usr/local/lib/python3.12/dist-packages/torch/nested/_internal/nested_tensor.py", line 7, in <module>
    from torch.nested._internal.nested_int import NestedIntNode
  File "/usr/local/lib/python3.12/dist-packages/torch/nested/_internal/nested_int.py", line 4, in <module>
    from torch.fx.experimental._constant_symnode import ConstantIntNode
  File "/usr/local/lib/python3.12/dist-packages/torch/fx/__init__.py", line 88, in <module>
    from torch.fx._symbolic_trace import (  # noqa: F401
  File "/usr/local/lib/python3.12/dist-packages/torch/fx/_symbolic_trace.py", line 24,

In [29]:
show_evaluation_outputs("results/rgb_multitask_2_1")

Metrics file not found: results/rgb_multitask_2_1/metrics.json


### Multi-task Baseline, Learned Uncertainty Weighting

In [30]:
# Train the joint RGB multi-task baseline with Learned Uncertainty Weighting
!python src/train_RGB.py \
  --task multitask \
  --loss_weighting learned \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --checkpoint_name best_rgb_multitask_learned_weights.pt

Traceback (most recent call last):
  File "/content/REPO/src/train_RGB.py", line 9, in <module>
  File "/content/REPO/src/dataset.py", line 8, in <module>
    from torchvision import transforms
  File "/usr/local/lib/python3.12/dist-packages/torchvision/__init__.py", line 8, in <module>
    from torchvision import _meta_registrations, datasets, io, models, ops, transforms, utils  # usort:skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchvision/datasets/__init__.py", line 1, in <module>
    from ._optical_flow import FlyingChairs, FlyingThings3D, HD1K, KittiFlow, Sintel
  File "/usr/local/lib/python3.12/dist-packages/torchvision/datasets/_optical_flow.py", line 13, in <module>
    from .folder import default_loader
  File "/usr/local/lib/python3.12/dist-packages/torchvision/datasets/folder.py", line 53, in <module>
    is_valid_file: Optional[Callable[[str], bool]] = None,
             

In [31]:
# Evaluate the joint RGB multi-task baseline with Learned Uncertainty Weighting
!python src/evaluate_RGB.py \
  --task multitask \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --checkpoint checkpoints/best_rgb_multitask_learned_weights.pt \
  --output_dir results/rgb_multitask_learned_weights \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

Traceback (most recent call last):
  File "/content/REPO/src/evaluate_RGB.py", line 7, in <module>
    import torch
  File "/usr/local/lib/python3.12/dist-packages/torch/__init__.py", line 2167, in <module>
    _C._initExtension(_manager_path())
  File "/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py", line 61, in <module>
    import pynvml  # type: ignore[import]
    ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pynvml.py", line 2894, in <module>
    class c_nvmlConfComputeSystemCaps_t(Structure):
  File "/usr/local/lib/python3.12/dist-packages/pynvml.py", line 2894, in c_nvmlConfComputeSystemCaps_t
    class c_nvmlConfComputeSystemCaps_t(Structure):
    
KeyboardInterrupt
^C


In [32]:
show_evaluation_outputs("results/rgb_multitask_learned_weights")

Metrics file not found: results/rgb_multitask_learned_weights/metrics.json


## Results Comparison and Ablation Study

In [33]:
results = {
    "RGB fake-only": "results/rgb_fake/metrics.json",
    "RGB transform-only": "results/rgb_transform/metrics.json",
    "RGB multitask 1-1": "results/rgb_multitask_1_1/metrics.json",
    "RGB multitask 1-2": "results/rgb_multitask_1_2/metrics.json",
    "RGB multitask 2-1": "results/rgb_multitask_2_1/metrics.json",
    "RGB multitask learned": "results/rgb_multitask_learned_weights/metrics.json",
}

rows = []

for model_name, metrics_path in results.items():
    metrics_path = Path(metrics_path)

    if not metrics_path.exists():
        print(f"Missing file: {metrics_path}")
        continue

    with open(metrics_path, "r") as f:
        metrics = json.load(f)

    rows.append({
        "model": model_name,
        "fake_accuracy": metrics.get("fake_accuracy"),
        "fake_f1_macro": metrics.get("fake_f1_macro"),
        "transform_accuracy": metrics.get("transform_accuracy"),
        "transform_f1_macro": metrics.get("transform_f1_macro"),
        "fake_acc_original": metrics.get("fake_accuracy_by_transform", {}).get("original"),
        "fake_acc_transfer": metrics.get("fake_accuracy_by_transform", {}).get("transfer"),
        "fake_acc_redigital": metrics.get("fake_accuracy_by_transform", {}).get("redigital"),
    })

comparison_df = pd.DataFrame(rows)
comparison_df.sort_values(
    by=["fake_accuracy", "transform_accuracy"],
    ascending=False
)

Missing file: results/rgb_fake/metrics.json
Missing file: results/rgb_transform/metrics.json
Missing file: results/rgb_multitask_1_1/metrics.json
Missing file: results/rgb_multitask_1_2/metrics.json
Missing file: results/rgb_multitask_2_1/metrics.json
Missing file: results/rgb_multitask_learned_weights/metrics.json


KeyError: 'fake_accuracy'

In [ ]:
# Global Accuracy Comparison

plot_df = comparison_df.set_index("model")

plot_df[["fake_accuracy", "transform_accuracy"]].plot(
    kind="bar",
    figsize=(12, 5),
)

plt.title("Global Accuracy Comparison")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# Real/Fake Accuracy by Transformation Type
plot_df[[
    "fake_acc_original",
    "fake_acc_transfer",
    "fake_acc_redigital",
]].plot(
    kind="bar",
    figsize=(12, 5),
)

plt.title("Real/Fake Accuracy by Transformation Type")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y")
plt.tight_layout()
plt.show()

#FREQUENCY


**TRAIN**

In [ ]:
# Frequency multi-task using weights 1.0 / 1.0
!python src/train_freq.py \
  --train_csv        {TRAIN_CSV} \
  --val_csv          {VAL_CSV} \
  --image_root       {IMAGE_ROOT} \
  --epochs           {EPOCHS} \
  --batch_size       {BATCH_SIZE} \
  --image_size       {IMAGE_SIZE} \
  --num_workers      {NUM_WORKERS} \
  --lambda_fake      1.0 \
  --lambda_transform 1.0 \
  --checkpoint_name  best_freq_multitask_1_1.pt

**EVALUATE**

In [ ]:
!python src/evaluate_freq.py \
  --csv_path    {TEST_CSV} \
  --image_root  {IMAGE_ROOT} \
  --checkpoint  checkpoints/best_freq_multitask_1_1.pt \
  --output_dir  results/freq_multitask_1_1 \
  --batch_size  {BATCH_SIZE} \
  --image_size  {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

**RESULTS**

In [ ]:
show_evaluation_outputs("results/freq_multitask_1_1")

**COMPARISON**

In [ ]:
results["Freq multitask 1-1"] = "results/freq_multitask_1_1/metrics.json"

#DEPTH

In [ ]:
!find /content -name "real_006970.jpg"

##Depth map generation

###first small debug

In [34]:
!python src/generate_depth_map.py \
  --csv_paths {TRAIN_CSV} {VAL_CSV} {TEST_CSV}\
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --model_type MiDaS_small \
  --max_images 5

Using device: cuda
Number of images to process: 5
Loading pretrained MiDaS model: MiDaS_small
Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Loading weights:  None
Downloading: "https://github.com/rwightman/gen-efficientnet-pytorch/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/tf_efficientnet_lite3-b733e338.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientnet_lite3-b733e338.pth
Downloading: "https://github.com/isl-org/MiDaS/releases/download/v2_1/midas_v21_small_256.pt" to /root/.cache/torch/hub/checkpoints/midas_v21_small_256.pt
100% 81.8M/81.8M [00:01<00

###full depth map generation

In [35]:
!python src/generate_depth_map.py \
  --csv_paths {TRAIN_CSV} {VAL_CSV} {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --model_type MiDaS_small

Using device: cuda
Number of images to process: 3000
Loading pretrained MiDaS model: MiDaS_small
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
Loading weights:  None
Using cache found in /root/.cache/torch/hub/rwightman_gen-efficientnet-pytorch_master
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master
Generating depth maps: 100% 3000/3000 [03:24<00:00, 14.65it/s]
Depth map generation completed.
Depth maps saved in: /content/drive/MyDrive/CV_Project/depth_maps


##copy of depth maph in local so colab doesn't need to go on drive every time

In [36]:
DRIVE_DEPTH_ROOT = "/content/drive/MyDrive/CV_Project/depth_maps"
LOCAL_DEPTH_ROOT = "/content/depth_maps"

!mkdir -p "{LOCAL_DEPTH_ROOT}"
!rsync -a --info=progress2 "{DRIVE_DEPTH_ROOT}/" "{LOCAL_DEPTH_ROOT}/"

DEPTH_ROOT = LOCAL_DEPTH_ROOT

print("Depth maps will be loaded from:", DEPTH_ROOT)

 15,795,119,360 100%  104.15MB/s    0:02:24 (xfr#3000, to-chk=0/3010)
Depth maps will be loaded from: /content/depth_maps


##First try with depth only

In [ ]:
!python src/train_depth.py \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint_name best_depth_uncertainty.pt \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lr {LEARNING_RATE} \
  --use_uncertainty_weighting \
  --no_edge

The edge-consistency branch introduces additional geometric information, but also increases model complexity. This leads to mild overfitting and a small decrease in real/fake validation accuracy, although transformation classification slightly improves.
We should use only depth without edge

##Evaluate RGB+Depth no edge

saves on drive not only local

In [ ]:
DRIVE_CKPT_ROOT = "/content/drive/MyDrive/CV_Project/checkpoints"

!mkdir -p "{DRIVE_CKPT_ROOT}"
!cp /content/REPO/checkpoints/*.pt "{DRIVE_CKPT_ROOT}/"

!ls "{DRIVE_CKPT_ROOT}"

puts again in local

In [ ]:
DRIVE_CKPT_ROOT = "/content/drive/MyDrive/CV_Project/checkpoints"
LOCAL_CKPT_ROOT = "/content/REPO/checkpoints"

!mkdir -p "{LOCAL_CKPT_ROOT}"
!cp "{DRIVE_CKPT_ROOT}"/*.pt "{LOCAL_CKPT_ROOT}/"

!ls "{LOCAL_CKPT_ROOT}"

In [ ]:
!python src/evaluate_depth.py \
  --task multitask \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_uncertainty.pt \
  --output_dir results/depth_only \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --no_edge

In [ ]:
show_evaluation_outputs("results/depth_only")

##grad-cam

real

In [ ]:
!python src/gradCam_depth.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_uncertainty.pt \
  --output_dir results/gradcam_depth_only \
  --task fake \
  --target predicted \
  --max_images 2 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --no_edge \
  --fake_filter 0 \
  --shuffle

In [ ]:
!python src/gradCam_depth.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_uncertainty.pt \
  --output_dir results/gradcam_depth_only \
  --task fake \
  --target predicted \
  --max_images 2 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --no_edge \
  --fake_filter 1 \
  --shuffle

In [ ]:
from IPython.display import Image, display
from pathlib import Path

gradcam_dir = Path("results/gradcam_depth_only")

for img_path in sorted(gradcam_dir.glob("*.png")):
    print(img_path.name)
    display(Image(filename=str(img_path)))

transform

In [ ]:
!python src/gradCam_depth.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_uncertainty.pt \
  --output_dir results/gradcam_transform \
  --task transform \
  --target predicted \
  --max_images 2 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --no_edge \
  --transform_filter 0 \
  --shuffle

In [ ]:
!python src/gradCam_depth.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_uncertainty.pt \
  --output_dir results/gradcam_transform \
  --task transform \
  --target predicted \
  --max_images 2 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --no_edge \
  --transform_filter 1 \
  --shuffle

In [ ]:
!python src/gradCam_depth.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_uncertainty.pt \
  --output_dir results/gradcam_transform \
  --task transform \
  --target predicted \
  --max_images 2 \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --no_edge \
  --transform_filter 2 \
  --shuffle

In [ ]:
from IPython.display import Image, display
from pathlib import Path

gradcam_dir = Path("results/gradcam_transform")

for img_path in sorted(gradcam_dir.glob("*.png")):
    print(img_path.name)
    display(Image(filename=str(img_path)))

##Second try with also edge consistency

In [ ]:
!python src/train_depth.py \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint_name best_depth_edge_uncertainty.pt \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lr {LEARNING_RATE} \
  --use_uncertainty_weighting

#Evaluate RGB+Depth+edge

In [ ]:
!python src/evaluate_depth.py \
  --task multitask \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_edge_uncertainty.pt \
  --output_dir results/depth_edge \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

In [ ]:
show_evaluation_outputs("results/depth_edge")

#DEPTH+FREQUENCY

In [ ]:
print(EPOCHS)
print(type(EPOCHS))

!echo {EPOCHS}

In [38]:
!python src/train_depth_frequency.py \
  --train_csv {TRAIN_CSV} \
  --val_csv {VAL_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint_name best_depth_frequency_uncertainty.pt \
   --checkpoint_dir checkpoints/best_depth_frequency_uncertainty.pt \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --image_size {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS} \
  --lr {LEARNING_RATE} \
  --use_uncertainty_weighting

Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 111MB/s]

Epoch 1/10
Training RGB+Depth+Frequency: 100% 66/66 [01:42<00:00,  1.55s/it]
Validation RGB+Depth+Frequency: 100% 15/15 [00:26<00:00,  1.80s/it]
Train: {'loss': 1.4172382561365764, 'fake_acc': 0.83, 'transform_acc': 0.43, 'weight_fake': 1.0035392569773125, 'weight_transform': 0.9972187136158799}
Val:   {'loss': 1.1241603048642477, 'fake_acc': 0.9, 'transform_acc': 0.5844444444444444, 'weight_fake': 1.0075130462646484, 'weight_transform': 0.996167778968811}
Val score: 0.7422
Learning rate: 0.000100
Saved best checkpoint to checkpoints/best_depth_frequency_uncertainty.pt/best_depth_frequency_uncertainty.pt

Epoch 2/10
Training RGB+Depth+Frequency: 100% 66/66 [01:41<00:00,  1.54s/it]
Validation RGB+Depth+Frequency: 100% 15/15 [00:27<00:00,  1.80s/it]
Train: {'loss': 0.85649818670182, 'fake_acc': 0.9

In [39]:
DRIVE_CKPT_ROOT = "/content/drive/MyDrive/CV_Project/checkpoints"

!mkdir -p "{DRIVE_CKPT_ROOT}"
!cp /content/REPO/checkpoints/*.pt "{DRIVE_CKPT_ROOT}/"

!ls "{DRIVE_CKPT_ROOT}"

cp: -r not specified; omitting directory '/content/REPO/checkpoints/best_depth_frequency_uncertainty.pt'


In [40]:
DRIVE_CKPT_ROOT = "/content/drive/MyDrive/CV_Project/checkpoints"
LOCAL_CKPT_ROOT = "/content/REPO/checkpoints"

!mkdir -p "{LOCAL_CKPT_ROOT}"
!cp "{DRIVE_CKPT_ROOT}"/*.pt "{LOCAL_CKPT_ROOT}/"

!ls "{LOCAL_CKPT_ROOT}"

cp: cannot stat '/content/drive/MyDrive/CV_Project/checkpoints/*.pt': No such file or directory
best_depth_frequency_uncertainty.pt


evaluate

In [42]:
!python src/evaluation_depth_frequency.py \
  --csv_path {TEST_CSV} \
  --image_root {IMAGE_ROOT} \
  --depth_root {DEPTH_ROOT} \
  --checkpoint checkpoints/best_depth_frequency_uncertainty.pt \
  --output_dir results/depth_frequency_uncertainty \
  --batch_size {BATCH_SIZE} \
  --image_size  {IMAGE_SIZE} \
  --num_workers {NUM_WORKERS}

Using device: cuda
Traceback (most recent call last):
  File "/content/REPO/src/evaluation_depth_frequency.py", line 321, in <module>
    main()
  File "/content/REPO/src/evaluation_depth_frequency.py", line 256, in main
    model = load_model(
            ^^^^^^^^^^^
  File "/content/REPO/src/evaluation_depth_frequency.py", line 47, in load_model
    model = RGBDepthFrequencyMultiTaskModel(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: RGBDepthFrequencyMultiTaskModel.__init__() got an unexpected keyword argument 'num_fake_classes'
